# Baseline MoE continual-learning experiments

This notebook runs the pre-registered baseline protocol. Cross-validation remains continual learning: each raw CIL task is stratified into three folds; corresponding task folds are held out together. Final replicas train task-by-task on all selected `train.csv` records. No training or validation prediction caches are saved.

Use a stable run name and resume setting in Colab. Baseline conditions use MobileNetV3 Large or ConvNeXt Tiny with either a single-layer regression router or a deeper MLP router.

In [ ]:
! git clone "https://github.com/ddimpfel/JHU_IS_26.git"
! git rev-parse HEAD

Cloning into 'JHU_IS_26'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 124 (delta 80), reused 100 (delta 56), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 33.89 MiB | 18.81 MiB/s, done.
Resolving deltas: 100% (80/80), done.


In [2]:
import os
os.chdir("/content/JHU_IS_26")
! pwd

/content/JHU_IS_26


In [3]:
! pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 52.9 MB/s eta 0:00:00


In [4]:
import pandas as pd
from IPython.display import display

from init_experiment import (
    ExperimentConfig,
    ExperimentRunner,
    build_gtsrb_data,
    build_model_specs,
    environment_summary,
    prepare_notebook_runtime,
)

# Runtime and output configuration
RESULTS_NAMESPACE = 'baseline_experiments'
RUN_NAME = 'baseline_cil_protocol'  # Stable directory name; change only to begin a new run.
RESUME = True
FINAL_RUN_SEEDS = (7, 17, 27, 37)

runtime = prepare_notebook_runtime(RESULTS_NAMESPACE)

# Reproducibility and data
SEED = 7
DATASET_NAME = 'meowmeowmeowmeowmeow/gtsrb-german-traffic-sign'
CLASS_IDS = (1, 2, 3, 4, 5, 7, 8, 9, 10)
IMAGE_SIZE = 224
BATCH_SIZE = 128
NUM_WORKERS = 4 if runtime.in_colab else 0
PIN_MEMORY = runtime.in_colab
PERSISTENT_WORKERS = runtime.in_colab

# Continual-learning optimization
NUM_TASKS = 3
CV_FOLDS = 3
CV_REPEATS = 1
CV_EPOCHS = 8
EPOCHS = 10  # Final full-source CIL epochs per task.
THROUGHPUT_WARMUP_BATCHES = 3
EXEMPLAR_RATIO = 0.065
TASK_1_LR = 0.001
LATER_TASK_LR_FACTOR = 0.1
USE_CLASS_MASKING = True
KD_TEMPERATURE = 2.0
LAMBDA_KD = 0.5

# Mixture-of-experts architecture
NUM_EXPERTS = 6
HIDDEN_EXPERT_SIZE = 128
DROPOUT = 0.1
TOP_K = 2
LAMBDA_AUX = 0.05
TRANSFORMER_D_MODEL = 32
TRANSFORMER_NHEAD = 4

# Joint-embedding values are recorded for traceability but unused by baseline conditions.
JE_EMBEDDING_DIM = 256
JE_PROJECTION_DIM = 256
JE_FEATURE_KEY = 'projections'  # 'embeddings' or 'projections'
JE_TEMPERATURE = 0.07
JE_CONTRASTIVE_WEIGHT = 0.1

# Execution
PRETRAINED_BACKBONES = True
DEVICE = None  # Set to 'cuda' or 'cpu' to override automatic selection.

config = ExperimentConfig(
    seed=SEED,
    cv_folds=CV_FOLDS,
    cv_repeats=CV_REPEATS,
    cv_epochs=CV_EPOCHS,
    final_run_seeds=FINAL_RUN_SEEDS,
    throughput_warmup_batches=THROUGHPUT_WARMUP_BATCHES,
    dataset_name=DATASET_NAME,
    class_ids=CLASS_IDS,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    num_tasks=NUM_TASKS,
    epochs=EPOCHS,
    exemplar_ratio=EXEMPLAR_RATIO,
    task_1_lr=TASK_1_LR,
    later_task_lr_factor=LATER_TASK_LR_FACTOR,
    use_class_masking=USE_CLASS_MASKING,
    kd_temperature=KD_TEMPERATURE,
    lambda_kd=LAMBDA_KD,
    num_experts=NUM_EXPERTS,
    hidden_expert_size=HIDDEN_EXPERT_SIZE,
    dropout=DROPOUT,
    top_k=TOP_K,
    lambda_aux=LAMBDA_AUX,
    transformer_d_model=TRANSFORMER_D_MODEL,
    transformer_nhead=TRANSFORMER_NHEAD,
    je_embedding_dim=JE_EMBEDDING_DIM,
    je_projection_dim=JE_PROJECTION_DIM,
    je_feature_key=JE_FEATURE_KEY,
    je_temperature=JE_TEMPERATURE,
    je_contrastive_weight=JE_CONTRASTIVE_WEIGHT,
    pretrained_backbones=PRETRAINED_BACKBONES,
    device=DEVICE,
 )

Mounted at /content/drive


In [5]:
data = build_gtsrb_data(config)
runner = ExperimentRunner(config, data, runtime.results_dir, run_name=RUN_NAME, resume=RESUME)

Using Colab cache for faster access to the 'gtsrb-german-traffic-sign' dataset.


In [6]:
display(pd.DataFrame([environment_summary(config)]))
display(pd.DataFrame({'Task': range(1, config.num_tasks + 1), 'Raw GTSRB classes': data.task_classes}))

,device,cuda_available,cuda_device_count,torch,torchvision,cuda_device_name
0,cuda,True,1,2.11.0+cu128,0.26.0+cu128,NVIDIA A100-SXM4-40GB


,Task,Raw GTSRB classes
0,1,"(10, 1, 9)"
1,2,"(2, 4, 8)"
2,3,"(3, 5, 7)"


In [7]:
baseline_specs = build_model_specs(config, family='baseline')
pd.DataFrame([spec.__dict__ for spec in baseline_specs.values()])

,name,family,backbone,router,expert,feature_space,uses_joint_embedding
0,MobileNet Large + Regression Router + MLP Experts,Baseline,MobileNet Large,Regression Router,MLP Experts,backbone_features,False
1,MobileNet Large + Regression Router + Transfor...,Baseline,MobileNet Large,Regression Router,Transformer Experts,backbone_features,False
2,MobileNet Large + Regression Router + ResMLP E...,Baseline,MobileNet Large,Regression Router,ResMLP Experts,backbone_features,False
3,MobileNet Large + MLP Router + MLP Experts,Baseline,MobileNet Large,MLP Router,MLP Experts,backbone_features,False
4,MobileNet Large + MLP Router + Transformer Exp...,Baseline,MobileNet Large,MLP Router,Transformer Experts,backbone_features,False
5,MobileNet Large + MLP Router + ResMLP Experts,Baseline,MobileNet Large,MLP Router,ResMLP Experts,backbone_features,False
6,ConvNeXt Tiny + Regression Router + MLP Experts,Baseline,ConvNeXt Tiny,Regression Router,MLP Experts,backbone_features,False
7,ConvNeXt Tiny + Regression Router + Transforme...,Baseline,ConvNeXt Tiny,Regression Router,Transformer Experts,backbone_features,False
8,ConvNeXt Tiny + Regression Router + ResMLP Exp...,Baseline,ConvNeXt Tiny,Regression Router,ResMLP Experts,backbone_features,False
9,ConvNeXt Tiny + MLP Router + MLP Experts,Baseline,ConvNeXt Tiny,MLP Router,MLP Experts,backbone_features,False


## CIL cross-validation

For every fold, each CIL task trains on its two local training shards and validates on its corresponding held-out shard. Cross-validation does not save checkpoints.

In [8]:
cv_run = runner.run_baseline_cross_validation(verbose=False)
cv_comparison_df = cv_run.comparison_df
cv_comparison_df.head()


=== CV MobileNet Large + Regression Router + MLP Experts (repeat=1, fold=1) ===
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 232MB/s]


Stage 3/3: Model GeneralistRouterExperts
=== CV MobileNet Large + Regression Router + Transformer Experts (repeat=1, fold=1) ===
Stage 3/3: Model GeneralistRouterExperts
=== CV MobileNet Large + Regression Router + ResMLP Experts (repeat=1, fold=1) ===
Stage 3/3: Model GeneralistRouterExperts
=== CV MobileNet Large + MLP Router + MLP Experts (repeat=1, fold=1) ===
Stage 3/3: Model GeneralistRouterExperts
=== CV MobileNet Large + MLP Router + Transformer Experts (repeat=1, fold=1) ===
Stage 3/3: Model GeneralistRouterExperts
=== CV MobileNet Large + MLP Router + ResMLP Experts (repeat=1, fold=1) ===
Stage 3/3: Model GeneralistRouterExperts
=== CV ConvNeXt Tiny + Regression Router + MLP Experts (repeat=1, fold=1) ===
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 243MB/s] 


Stage 3/3: Model GeneralistRouterExperts
=== CV ConvNeXt Tiny + Regression Router + Transformer Experts (repeat=1, fold=1) ===
Stage 3/3: Model GeneralistRouterExperts
=== CV ConvNeXt Tiny + Regression Router + ResMLP Experts (repeat=1, fold=1) ===
Stage 3/3: Model GeneralistRouterExperts
=== CV ConvNeXt Tiny + MLP Router + MLP Experts (repeat=1, fold=1) ===
Stage 3/3: Model GeneralistRouterExperts
=== CV ConvNeXt Tiny + MLP Router + Transformer Experts (repeat=1, fold=1) ===
Stage 3/3: Model GeneralistRouterExperts
=== CV ConvNeXt Tiny + MLP Router + ResMLP Experts (repeat=1, fold=1) ===
Stage 3/3: Model GeneralistRouterExperts
=== CV MobileNet Large + Regression Router + MLP Experts (repeat=1, fold=2) ===
Stage 3/3: Model GeneralistRouterExperts
=== CV MobileNet Large + Regression Router + Transformer Experts (repeat=1, fold=2) ===
Stage 3/3: Model GeneralistRouterExperts
=== CV MobileNet Large + Regression Router + ResMLP Experts (repeat=1, fold=2) ===
Stage 3/3: Model GeneralistRou

,Stage,Model,Family,Backbone,Router,Expert,Feature Space,CV Repeat,CV Fold,Run Seed,...,Measured Optimization Time (s),Training Throughput (samples/s),Validation Router Entropy,Validation Normalized Router Entropy,Validation Router Entropy Std,Validation Avg Router Prob,Validation ECE,Validation Expected Expert Calls,Validation Expert Utilization Ratio,Validation Cost Proxy
0,cv,ConvNeXt Tiny + MLP Router + MLP Experts,Baseline,ConvNeXt Tiny,MLP Router,MLP Experts,backbone_features,1,1,7,...,326.376219,276.833895,1.085322,0.605730,0.556982,"[0.09216710180044174, 0.10456163436174393, 0.1...",0.006161,2.0,"[0.06570093333721161, 0.13719625771045685, 0.1...",30212565.0
1,cv,ConvNeXt Tiny + MLP Router + MLP Experts,Baseline,ConvNeXt Tiny,MLP Router,MLP Experts,backbone_features,1,2,8,...,327.207765,276.130366,1.307398,0.729673,0.469667,"[0.24102336168289185, 0.12641556560993195, 0.1...",0.015316,2.0,"[0.1780373901128769, 0.21186916530132294, 0.15...",30212565.0
2,cv,ConvNeXt Tiny + MLP Router + MLP Experts,Baseline,ConvNeXt Tiny,MLP Router,MLP Experts,backbone_features,1,3,9,...,328.936239,274.679373,0.848631,0.473630,0.640251,"[0.40082767605781555, 0.11275872588157654, 0.2...",0.014666,2.0,"[0.23084111511707306, 0.11841121315956116, 0.1...",30212565.0
3,cv,ConvNeXt Tiny + MLP Router + ResMLP Experts,Baseline,ConvNeXt Tiny,MLP Router,ResMLP Experts,backbone_features,1,1,7,...,331.766434,272.336170,0.220445,0.123032,0.334483,"[0.2961917221546173, 0.12670470774173737, 0.25...",0.043720,2.0,"[0.19252336025238037, 0.16841121017932892, 0.1...",31029717.0
4,cv,ConvNeXt Tiny + MLP Router + ResMLP Experts,Baseline,ConvNeXt Tiny,MLP Router,ResMLP Experts,backbone_features,1,2,8,...,332.317263,271.884762,1.133253,0.632480,0.589420,"[0.12895502150058746, 0.11899462342262268, 0.2...",0.003511,2.0,"[0.1619626134634018, 0.14411215484142303, 0.21...",31029717.0


In [9]:
cv_summary_columns = [
    'Model', 'CV Repeat', 'CV Fold', 'Backbone', 'Router', 'Expert',
    'AvgAcc Micro F1', 'Backward Transfer Micro F1',
    'Forward Transfer Micro F1', 'Average Forgetting Micro F1',
    'Full Validation Micro F1', 'Validation ECE', 'Training Throughput (samples/s)', 'Validation Cost Proxy',
]
cv_summary_df = cv_comparison_df.loc[:, [column for column in cv_summary_columns if column in cv_comparison_df.columns]]
cv_summary_df.sort_values(['AvgAcc Micro F1', 'Full Validation Micro F1'], ascending=False).reset_index(drop=True)

,Model,CV Repeat,CV Fold,Backbone,Router,Expert,AvgAcc Micro F1,Backward Transfer Micro F1,Forward Transfer Micro F1,Average Forgetting Micro F1,Full Validation Micro F1,Validation ECE,Training Throughput (samples/s),Validation Cost Proxy
0,ConvNeXt Tiny + Regression Router + Transforme...,1,1,ConvNeXt Tiny,Regression Router,Transformer Experts,0.965898,-0.045364,-0.170284,0.045364,0.964685,0.003356,266.425056,29496021.0
1,ConvNeXt Tiny + MLP Router + ResMLP Experts,1,2,ConvNeXt Tiny,MLP Router,ResMLP Experts,0.961580,-0.054656,-0.067431,0.054656,0.959615,0.003511,271.884762,31029717.0
2,ConvNeXt Tiny + Regression Router + ResMLP Exp...,1,2,ConvNeXt Tiny,Regression Router,ResMLP Experts,0.961569,-0.053611,-0.031231,0.053611,0.959846,0.009304,271.548633,29843925.0
3,ConvNeXt Tiny + MLP Router + MLP Experts,1,1,ConvNeXt Tiny,MLP Router,MLP Experts,0.961165,-0.051722,-0.130124,0.051722,0.960065,0.006161,276.833895,30212565.0
4,ConvNeXt Tiny + Regression Router + MLP Experts,1,1,ConvNeXt Tiny,Regression Router,MLP Experts,0.960903,-0.049286,-0.121512,0.049286,0.959460,0.006124,268.899519,29026773.0
5,ConvNeXt Tiny + Regression Router + MLP Experts,1,2,ConvNeXt Tiny,Regression Router,MLP Experts,0.956728,-0.062413,-0.065114,0.062413,0.954501,0.007969,276.023347,29026773.0
6,ConvNeXt Tiny + MLP Router + ResMLP Experts,1,3,ConvNeXt Tiny,MLP Router,ResMLP Experts,0.953392,-0.061520,-0.157445,0.061520,0.951588,0.004833,271.357700,31029717.0
7,ConvNeXt Tiny + MLP Router + MLP Experts,1,3,ConvNeXt Tiny,MLP Router,MLP Experts,0.953319,-0.054894,-0.145402,0.054894,0.952127,0.014666,274.679373,30212565.0
8,ConvNeXt Tiny + Regression Router + Transforme...,1,3,ConvNeXt Tiny,Regression Router,Transformer Experts,0.947057,-0.072674,-0.078496,0.072674,0.945217,0.013035,266.284346,29496021.0
9,MobileNet Large + Regression Router + Transfor...,1,2,MobileNet Large,Regression Router,Transformer Experts,0.940227,-0.080946,-0.140121,0.080946,0.936774,0.093379,910.500007,4653047.0


## Final full-source CIL replicas

Run after the CV selection decision is fixed. This stage saves one checkpoint per selected condition and final seed.

In [10]:
final_run = runner.run_baseline_final(verbose=False)
final_comparison_df = final_run.comparison_df
final_comparison_df.head()


=== Final MobileNet Large + Regression Router + MLP Experts (seed=7) ===

=== Final MobileNet Large + Regression Router + Transformer Experts (seed=7) ===

=== Final MobileNet Large + Regression Router + ResMLP Experts (seed=7) ===

=== Final MobileNet Large + MLP Router + MLP Experts (seed=7) ===

=== Final MobileNet Large + MLP Router + Transformer Experts (seed=7) ===

=== Final MobileNet Large + MLP Router + ResMLP Experts (seed=7) ===

=== Final ConvNeXt Tiny + Regression Router + MLP Experts (seed=7) ===

=== Final ConvNeXt Tiny + Regression Router + Transformer Experts (seed=7) ===

=== Final ConvNeXt Tiny + Regression Router + ResMLP Experts (seed=7) ===

=== Final ConvNeXt Tiny + MLP Router + MLP Experts (seed=7) ===

=== Final ConvNeXt Tiny + MLP Router + Transformer Experts (seed=7) ===

=== Final ConvNeXt Tiny + MLP Router + ResMLP Experts (seed=7) ===

=== Final MobileNet Large + Regression Router + MLP Experts (seed=17) ===

=== Final MobileNet Large + Regression Router 

,Stage,Model,Family,Backbone,Router,Expert,Feature Space,Final Seed,Training Throughput per Stage (samples/s),Training Samples per Stage,...,Training Throughput (samples/s),Final Exemplar Count,Training Router Entropy,Training Normalized Router Entropy,Training Router Entropy Std,Training Avg Router Prob,Training ECE,Training Expected Expert Calls,Training Expert Utilization Ratio,Training Cost Proxy
0,final,ConvNeXt Tiny + MLP Router + MLP Experts,Baseline,ConvNeXt Tiny,MLP Router,MLP Experts,backbone_features,17,"[317.2138986490066, 257.81840588618604, 257.70...","[57000, 60100, 54460]",...,274.881689,1042,1.541279,0.860204,0.286110,"[0.24242284893989563, 0.13507628440856934, 0.1...",0.005531,2.0,"[0.23255451023578644, 0.16323988139629364, 0.1...",30212565.0
1,final,ConvNeXt Tiny + MLP Router + MLP Experts,Baseline,ConvNeXt Tiny,MLP Router,MLP Experts,backbone_features,27,"[316.64695179281557, 258.59814978504573, 257.9...","[57000, 60100, 54460]",...,275.141001,1042,1.532182,0.855127,0.191658,"[0.19125722348690033, 0.09091773629188538, 0.2...",0.011584,2.0,"[0.21411214768886566, 0.053021807223558426, 0....",30212565.0
2,final,ConvNeXt Tiny + MLP Router + MLP Experts,Baseline,ConvNeXt Tiny,MLP Router,MLP Experts,backbone_features,37,"[316.17038117660957, 258.16166446330226, 257.1...","[57000, 60100, 54460]",...,274.557760,1042,0.990633,0.552883,0.535745,"[0.1969643384218216, 0.13153403997421265, 0.23...",0.011682,2.0,"[0.18121495842933655, 0.15828660130500793, 0.1...",30212565.0
3,final,ConvNeXt Tiny + MLP Router + MLP Experts,Baseline,ConvNeXt Tiny,MLP Router,MLP Experts,backbone_features,7,"[321.6028532644431, 258.23128020240046, 258.21...","[57000, 60100, 54460]",...,276.317033,1042,1.686956,0.941508,0.107135,"[0.1505090445280075, 0.1661214679479599, 0.178...",0.001775,2.0,"[0.08990654349327087, 0.23965732753276825, 0.1...",30212565.0
4,final,ConvNeXt Tiny + MLP Router + ResMLP Experts,Baseline,ConvNeXt Tiny,MLP Router,ResMLP Experts,backbone_features,17,"[312.25418844365925, 253.6533191171323, 253.30...","[57000, 60100, 54460]",...,270.395076,1042,0.529048,0.295267,0.497221,"[0.16796720027923584, 0.16617126762866974, 0.1...",0.012571,2.0,"[0.09218068420886993, 0.1673208773136139, 0.16...",31029717.0


In [11]:
final_summary_columns = [
    'Model', 'Final Seed', 'Backbone', 'Router', 'Expert',
    'Training Throughput (samples/s)', 'Training Cost Proxy', 'Num Parameters',
]
final_summary_df = final_comparison_df.loc[:, [column for column in final_summary_columns if column in final_comparison_df.columns]]
final_summary_df.sort_values(['Model', 'Final Seed']).reset_index(drop=True)

,Model,Final Seed,Backbone,Router,Expert,Training Throughput (samples/s),Training Cost Proxy
0,ConvNeXt Tiny + MLP Router + MLP Experts,7,ConvNeXt Tiny,MLP Router,MLP Experts,276.317033,30212565.0
1,ConvNeXt Tiny + MLP Router + MLP Experts,17,ConvNeXt Tiny,MLP Router,MLP Experts,274.881689,30212565.0
2,ConvNeXt Tiny + MLP Router + MLP Experts,27,ConvNeXt Tiny,MLP Router,MLP Experts,275.141001,30212565.0
3,ConvNeXt Tiny + MLP Router + MLP Experts,37,ConvNeXt Tiny,MLP Router,MLP Experts,274.557760,30212565.0
4,ConvNeXt Tiny + MLP Router + ResMLP Experts,7,ConvNeXt Tiny,MLP Router,ResMLP Experts,270.629749,31029717.0
5,ConvNeXt Tiny + MLP Router + ResMLP Experts,17,ConvNeXt Tiny,MLP Router,ResMLP Experts,270.395076,31029717.0
6,ConvNeXt Tiny + MLP Router + ResMLP Experts,27,ConvNeXt Tiny,MLP Router,ResMLP Experts,270.582127,31029717.0
7,ConvNeXt Tiny + MLP Router + ResMLP Experts,37,ConvNeXt Tiny,MLP Router,ResMLP Experts,270.903932,31029717.0
8,ConvNeXt Tiny + MLP Router + Transformer Experts,7,ConvNeXt Tiny,MLP Router,Transformer Experts,264.351192,30681813.0
9,ConvNeXt Tiny + MLP Router + Transformer Experts,17,ConvNeXt Tiny,MLP Router,Transformer Experts,264.737948,30681813.0
